# Data Cleaning and Normalization for MOTM Prediction
## Làm sạch và chuẩn hóa dữ liệu cho dự đoán MOTM

Notebook này hướng dẫn chi tiết cách làm sạch và chuẩn hóa dữ liệu cho dự án dự đoán Man of The Match

## BLOCK 1: Cài đặt và Import thư viện

In [ ]:
# Import các thư viện cần thiết
import pandas as pd
import numpy as np
import re
import warnings
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Cấu hình hiển thị
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ Tất cả thư viện đã được import thành công!")
print(f"✓ Pandas version: {pd.__version__}")
print(f"✓ NumPy version: {np.__version__}")

## BLOCK 2: Mount Google Drive (Nếu dùng Colab)

In [ ]:
# Kiểm tra và mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    colab = True
    print("✓ Google Drive đã được mount thành công!")
except:
    colab = False
    print("✓ Không phải Google Colab - chạy local")

## BLOCK 3: Cấu hình đường dẫn file

In [ ]:
# ============ CẤU HÌNH ĐƯỜNG DẪN ============

if colab:
    # Đối với Google Colab - điều chỉnh path của bạn
    base_path = '/content/drive/My Drive/MotmPrediction'
    input_file = f'{base_path}/PlayerCrawl.xlsx'
    output_clean = f'{base_path}/PlayerCrawl_cleaned.xlsx'
    output_normalized = f'{base_path}/PlayerCrawl_normalized.xlsx'
    output_clean_csv = f'{base_path}/PlayerCrawl_cleaned.csv'
else:
    # Đối với chạy local
    base_path = '.'
    input_file = 'PlayerCrawl.xlsx'
    output_clean = 'PlayerCrawl_cleaned.xlsx'
    output_normalized = 'PlayerCrawl_normalized.xlsx'
    output_clean_csv = 'PlayerCrawl_cleaned.csv'

print(f"📁 Đường dẫn input: {input_file}")
print(f"📁 Đường dẫn output (clean): {output_clean}")
print(f"📁 Đường dẫn output (normalized): {output_normalized}")
print(f"📁 Đường dẫn output (CSV): {output_clean_csv}")

## BLOCK 4: Load dữ liệu gốc

In [ ]:
# Load dữ liệu
df = pd.read_excel(input_file)

# Lưu bản sao để so sánh sau
df_original = df.copy()

print("="*70)
print("THÔNG TIN TỔNG QUÁT DỮ LIỆU GỐC")
print("="*70)
print(f"\n📊 Kích thước dữ liệu:")
print(f"   • Số dòng: {len(df):,}")
print(f"   • Số cột: {len(df.columns)}")
print(f"   • Tổng cells: {len(df) * len(df.columns):,}")

print(f"\n📋 Danh sách các cột:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:2d}. {col}")

print(f"\n💾 Kiểu dữ liệu:")
print(df.dtypes)

print(f"\n📈 Dữ liệu mẫu (5 dòng đầu):")
print(df.head())

print(f"\n📊 Thống kê mô tả:")
print(df.describe())

## BLOCK 5: Phân tích dữ liệu - Missing Values

In [ ]:
# Phân tích missing values
print("="*70)
print("PHÂN TÍCH MISSING VALUES (GIẢI TRỊ THIẾU)")
print("="*70)

missing_stats = pd.DataFrame({
    'Cột': df.columns,
    'Kiểu': df.dtypes.values,
    'Missing Count': df.isnull().sum().values,
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2).values,
    'Non-Null Count': df.count().values
}).sort_values('Missing %', ascending=False)

print("\n" + missing_stats.to_string(index=False))

# Tóm tắt
total_missing = df.isnull().sum().sum()
total_cells = len(df) * len(df.columns)
missing_percent = total_missing / total_cells * 100

print(f"\n📊 Tóm tắt:")
print(f"   • Tổng missing cells: {total_missing:,} / {total_cells:,}")
print(f"   • Tỷ lệ missing: {missing_percent:.2f}%")

# Visualize
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
missing_stats[missing_stats['Missing %'] > 0].plot(x='Cột', y='Missing %', kind='barh', color='coral')
plt.title('Tỷ lệ Missing Values theo Cột')
plt.xlabel('Missing %')

plt.subplot(1, 2, 2)
plt.text(0.5, 0.5, f'Tổng: {total_missing:,} / {total_cells:,}\n({missing_percent:.2f}%)', 
         ha='center', va='center', fontsize=16, transform=plt.gca().transAxes)
plt.axis('off')
plt.tight_layout()
plt.show()

## BLOCK 6: Phân tích dữ liệu - Duplicates (Trùng lặp)

In [ ]:
print("="*70)
print("PHÂN TÍCH DUPLICATES (DỮ LIỆU TRÙNG LẶP)")
print("="*70)

# Kiểm tra trùng lặp toàn bộ dòng
total_duplicates = df.duplicated().sum()
print(f"\n📊 Dòng trùng lặp toàn bộ: {total_duplicates}")

if total_duplicates > 0:
    print(f"\n❌ Cảnh báo: Phát hiện {total_duplicates} dòng trùng lặp!")
    print(f"\n📋 Ví dụ dòng trùng lặp (hiển thị 10 dòng đầu):")
    duplicates_df = df[df.duplicated(keep=False)].sort_values(by=list(df.columns))
    print(duplicates_df.head(10))
else:
    print(f"\n✓ Không phát hiện dòng trùng lặp")

# Kiểm tra trùng lặp từng cột
print(f"\n📊 Trùng lặp theo từng cột:")
print("-" * 70)
for col in df.columns:
    unique_count = df[col].nunique()
    total_count = len(df[col].dropna())
    duplicate_percent = (1 - unique_count / total_count) * 100 if total_count > 0 else 0
    print(f"   {col:20s}: {unique_count:6,} unique / {total_count:6,} total ({duplicate_percent:5.2f}% duplicate)")

## BLOCK 7: Định nghĩa các hàm làm sạch dữ liệu

In [ ]:
print("="*70)
print("ĐỊNH NGHĨA CÁC HÀM LÀM SẠCH DỮ LIỆU")
print("="*70)

def clean_text(text):
    """
    Làm sạch dữ liệu text:
    - Loại bỏ khoảng trắng dư thừa
    - Xóa ký tự đặc biệt
    - Chuẩn hóa định dạng
    """
    if pd.isna(text):
        return None
    
    text = str(text).strip()
    # Loại bỏ khoảng trắng dư thừa
    text = ' '.join(text.split())
    # Loại bỏ ký tự đặc biệt nếu cần (tuỳ chỉnh)
    # text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    
    return text if text else None

def clean_numeric(value):
    """
    Làm sạch dữ liệu số:
    - Loại bỏ ký tự không phải số
    - Chuyển đổi thành float
    """
    if pd.isna(value):
        return None
    
    try:
        # Loại bỏ ký tự không phải số (giữ lại dấu âm, dấu chấm, khoảng cách)
        cleaned = str(value).strip()
        cleaned = re.sub(r'[^0-9.-]', '', cleaned)
        
        if cleaned and cleaned != '.' and cleaned != '-':
            return float(cleaned)
        else:
            return None
    except:
        return None

def clean_date(date_value):
    """
    Làm sạch dữ liệu ngày tháng:
    - Chuẩn hóa format
    - Xử lý các định dạng khác nhau
    """
    if pd.isna(date_value):
        return None
    
    try:
        if isinstance(date_value, str):
            # Thử các định dạng ngày tháng phổ biến
            date_formats = ["%d/%m/%Y", "%Y-%m-%d", "%d-%m-%Y", "%d.%m.%Y", "%Y/%m/%d"]
            
            for fmt in date_formats:
                try:
                    return pd.to_datetime(date_value, format=fmt)
                except:
                    continue
            
            # Nếu không khớp format nào, để pandas tự nhận diện
            return pd.to_datetime(date_value)
        else:
            return pd.to_datetime(date_value)
    except:
        return None

def remove_outliers_iqr(series):
    """
    Phát hiện outliers sử dụng Interquartile Range (IQR)
    Trả về mask boolean để xác định outliers
    """
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    return (series >= lower_bound) & (series <= upper_bound)

print("\n✓ Các hàm sau đã được định nghĩa:")
print("   • clean_text() - Làm sạch text")
print("   • clean_numeric() - Làm sạch số")
print("   • clean_date() - Làm sạch ngày tháng")
print("   • remove_outliers_iqr() - Phát hiện outliers")

## BLOCK 8: Áp dụng làm sạch dữ liệu

In [ ]:
print("="*70)
print("ÁP DỤNG LÀM SẠCH DỮ LIỆU")
print("="*70)

# Tạo bản sao để làm sạch
df_clean = df.copy()

print(f"\n1️⃣ LÀM SẠCH CỘT TEXT")
print("-" * 70)
text_columns = [col for col in df_clean.columns if df_clean[col].dtype == 'object']
print(f"Các cột text: {text_columns}")

for col in text_columns:
    before_nulls = df_clean[col].isnull().sum()
    df_clean[col] = df_clean[col].apply(clean_text)
    after_nulls = df_clean[col].isnull().sum()
    print(f"   • {col}: {before_nulls} -> {after_nulls} missing values")

print(f"\n2️⃣ LOẠI BỎ DÒNG TRÙNG LẶP")
print("-" * 70)
before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates()
after_dedup = len(df_clean)
removed = before_dedup - after_dedup
print(f"   • Trước: {before_dedup:,} dòng")
print(f"   • Sau: {after_dedup:,} dòng")
print(f"   • Xóa: {removed:,} dòng ({removed/before_dedup*100:.2f}%)")

print(f"\n3️⃣ RESET INDEX")
print("-" * 70)
df_clean.reset_index(drop=True, inplace=True)
print(f"   ✓ Index đã được reset")

print(f"\n" + "="*70)
print("📊 TÓM TẮT KẾT QUẢ LÀM SẠCH")
print("="*70)
print(f"\nTrước làm sạch:")
print(f"   • Số dòng: {len(df):,}")
print(f"   • Số cột: {len(df.columns)}")
print(f"\nSau làm sạch:")
print(f"   • Số dòng: {len(df_clean):,}")
print(f"   • Số cột: {len(df_clean.columns)}")
print(f"   • Dòng được xóa: {len(df) - len(df_clean):,}")

## BLOCK 9: Xử lý Missing Values

In [ ]:
print("="*70)
print("XỬ LÝ MISSING VALUES")
print("="*70)

# ========== CHỌN MỘT TRONG CÁC CHIẾN LƯỢC SAU ==========

# CHIẾN LƯỢC 1: Loại bỏ dòng có bất kỳ missing value nào
# Phù hợp: khi dữ liệu đầy đủ là yêu cầu bắt buộc
print("\n🔹 CHIẾN LƯỢC 1: Loại bỏ dòng có missing values")
print("-" * 70)
df_strategy1 = df_clean.dropna()
print(f"   • Dòng trước: {len(df_clean):,}")
print(f"   • Dòng sau: {len(df_strategy1):,}")
print(f"   • Dòng xóa: {len(df_clean) - len(df_strategy1):,}")

# CHIẾN LƯỢC 2: Điền missing values bằng trung bình (cho cột số)
# Phù hợp: khi muốn giữ lại dòng nhưng điền giá trị số
print("\n🔹 CHIẾN LƯỢC 2: Điền missing bằng trung bình (số)")
print("-" * 70)
df_strategy2 = df_clean.copy()
numeric_cols = df_strategy2.select_dtypes(include=[np.number]).columns
print(f"   • Cột số: {list(numeric_cols)}")
for col in numeric_cols:
    before = df_strategy2[col].isnull().sum()
    df_strategy2[col].fillna(df_strategy2[col].mean(), inplace=True)
    after = df_strategy2[col].isnull().sum()
    print(f"   • {col}: điền {before} giá trị")

# CHIẾN LƯỢC 3: Điền missing values bằng 'Unknown' (cho cột text)
# Phù hợp: khi muốn đánh dấu dữ liệu thiếu với giá trị đặc biệt
print("\n🔹 CHIẾN LƯỢC 3: Điền missing bằng 'Unknown' (text)")
print("-" * 70)
df_strategy3 = df_clean.copy()
text_cols = df_strategy3.select_dtypes(include=['object']).columns
print(f"   • Cột text: {list(text_cols)}")
for col in text_cols:
    before = df_strategy3[col].isnull().sum()
    df_strategy3[col].fillna('Unknown', inplace=True)
    after = df_strategy3[col].isnull().sum()
    print(f"   • {col}: điền {before} giá trị")

# CHIẾN LƯỢC 4: Loại bỏ cột có quá nhiều missing values
# Phù hợp: khi cột có quá nhiều dữ liệu thiếu (không đáng tin cậy)
print("\n🔹 CHIẾN LƯỢC 4: Loại bỏ cột có quá nhiều missing")
print("-" * 70)
threshold = 0.5  # Nếu > 50% missing, loại bỏ
df_strategy4 = df_clean.copy()
cols_before = len(df_strategy4.columns)
cols_to_drop = [col for col in df_strategy4.columns 
                 if df_strategy4[col].isnull().sum() / len(df_strategy4) > threshold]
print(f"   • Threshold: {threshold*100:.0f}%")
print(f"   • Cột cần xóa: {cols_to_drop}")
df_strategy4 = df_strategy4.drop(columns=cols_to_drop)
print(f"   • Cột trước: {cols_before}")
print(f"   • Cột sau: {len(df_strategy4)}")

# ========== CHỌN CHIẾN LƯỢC CUỐI CÙNG ==========
print("\n" + "="*70)
print("💡 CHỌN CHIẾN LƯỢC CUỐI CÙNG")
print("="*70)
print("\nChọn một trong các chiến lược trên:")
print("  1 = Loại bỏ dòng có missing")
print("  2 = Điền trung bình (số) + Unknown (text)")
print("  3 = Chỉ điền 'Unknown' cho text")
print("  4 = Loại bỏ cột có quá nhiều missing")

# Mặc định: Chiến lược 2 (kết hợp)
strategy = 2
print(f"\n➜ Sử dụng chiến lược {strategy}")

if strategy == 1:
    df_clean = df_strategy1
elif strategy == 2:
    df_clean = df_strategy2.copy()
    text_cols = df_clean.select_dtypes(include=['object']).columns
    for col in text_cols:
        df_clean[col].fillna('Unknown', inplace=True)
elif strategy == 3:
    df_clean = df_strategy3
elif strategy == 4:
    df_clean = df_strategy4

print(f"\n✓ Áp dụng chiến lược {strategy} thành công!")
print(f"\nMissing values còn lại:")
print(df_clean.isnull().sum())

## BLOCK 10: Kiểm tra và làm sạch cột số

In [ ]:
print("="*70)
print("KIỂM TRA VÀ LÀM SẠCH CỘT SỐ")
print("="*70)

# Xác định cột số
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()

if numeric_cols:
    print(f"\nCác cột số được phát hiện: {numeric_cols}")
    
    for col in numeric_cols:
        print(f"\n📊 Cột: {col}")
        print("-" * 70)
        print(f"   • Kiểu: {df_clean[col].dtype}")
        print(f"   • Null: {df_clean[col].isnull().sum()}")
        print(f"   • Min: {df_clean[col].min():.4f}")
        print(f"   • Max: {df_clean[col].max():.4f}")
        print(f"   • Mean: {df_clean[col].mean():.4f}")
        print(f"   • Median: {df_clean[col].median():.4f}")
        print(f"   • Std: {df_clean[col].std():.4f}")
else:
    print("\n⚠️ Không có cột số")

## BLOCK 11: Phát hiện Outliers

In [ ]:
print("="*70)
print("PHÁT HIỆN OUTLIERS (GIÁ TRỊ NGOẠI LỆ)")
print("="*70)

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()

if numeric_cols:
    outliers_summary = {}
    
    for col in numeric_cols:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers_mask = (df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)
        num_outliers = outliers_mask.sum()
        outliers_summary[col] = {
            'Q1': Q1,
            'Q3': Q3,
            'IQR': IQR,
            'lower': lower_bound,
            'upper': upper_bound,
            'count': num_outliers,
            'percent': num_outliers / len(df_clean) * 100
        }
        
        print(f"\n📊 Cột: {col}")
        print("-" * 70)
        print(f"   • Q1 (25%): {Q1:.4f}")
        print(f"   • Q3 (75%): {Q3:.4f}")
        print(f"   • IQR: {IQR:.4f}")
        print(f"   • Khoảng bình thường: [{lower_bound:.4f}, {upper_bound:.4f}]")
        print(f"   • Số outliers: {num_outliers} ({outliers_summary[col]['percent']:.2f}%)")
        
        if num_outliers > 0:
            print(f"   • Giá trị outliers: {sorted(df_clean[col][outliers_mask].unique())}")
    
    # Visualize
    fig, axes = plt.subplots(1, len(numeric_cols), figsize=(15, 5))
    if len(numeric_cols) == 1:
        axes = [axes]
    
    for idx, col in enumerate(numeric_cols):
        axes[idx].boxplot(df_clean[col].dropna())
        axes[idx].set_title(f'{col}\n({outliers_summary[col]["count"]} outliers)')
        axes[idx].set_ylabel('Value')
    
    plt.tight_layout()
    plt.show()
    
    print("\n💡 Lưu ý: Chọn xóa outliers nếu bạn chắc chắn chúng là lỗi dữ liệu")
    print("   Để xóa outliers, hãy chạy code ở block tiếp theo")
else:
    print("\n⚠️ Không có cột số để phát hiện outliers")

## BLOCK 12: (Tuỳ chọn) Xóa Outliers

In [ ]:
print("="*70)
print("XÓA OUTLIERS (TỰY CHỌN)")
print("="*70)

# ========== CẤU HÌNH XÓA OUTLIERS ==========
REMOVE_OUTLIERS = False  # Đặt True để xóa, False để giữ
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()

print(f"\n⚙️ Cấu hình: REMOVE_OUTLIERS = {REMOVE_OUTLIERS}")

if REMOVE_OUTLIERS and numeric_cols:
    before_rows = len(df_clean)
    
    for col in numeric_cols:
        mask = remove_outliers_iqr(df_clean[col])
        df_clean = df_clean[mask]
    
    after_rows = len(df_clean)
    removed_rows = before_rows - after_rows
    
    print(f"\n✓ Dòng trước: {before_rows:,}")
    print(f"✓ Dòng sau: {after_rows:,}")
    print(f"✓ Dòng xóa: {removed_rows:,} ({removed_rows/before_rows*100:.2f}%)")
    
    df_clean.reset_index(drop=True, inplace=True)
elif REMOVE_OUTLIERS:
    print("\n⚠️ Không có cột số, không thể xóa outliers")
else:
    print("\n✓ Giữ lại outliers (REMOVE_OUTLIERS = False)")

## BLOCK 13: Chuẩn hóa dữ liệu (Normalization)

In [ ]:
print("="*70)
print("CHUẨN HÓA DỮ LIỆU (NORMALIZATION)")
print("="*70)

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()

if numeric_cols:
    print(f"\nCác cột sẽ được chuẩn hóa: {numeric_cols}")
    
    # ========== MIN-MAX SCALING ==========
    print(f"\n🔹 MIN-MAX SCALING (Range: [0, 1])")
    print("-" * 70)
    
    scaler_minmax = MinMaxScaler(feature_range=(0, 1))
    df_minmax = df_clean.copy()
    df_minmax[numeric_cols] = scaler_minmax.fit_transform(df_clean[numeric_cols])
    
    print(f"   • Công thức: (x - min) / (max - min)")
    print(f"   • Phạm vi: [0, 1]")
    print(f"\n   Thống kê sau Min-Max Scaling:")
    print(df_minmax[numeric_cols].describe())
    
    # ========== STANDARD SCALING ==========
    print(f"\n🔹 STANDARD SCALING (Standardization)")
    print("-" * 70)
    
    scaler_std = StandardScaler()
    df_standardized = df_clean.copy()
    df_standardized[numeric_cols] = scaler_std.fit_transform(df_clean[numeric_cols])
    
    print(f"   • Công thức: (x - mean) / std")
    print(f"   • Mean: 0, Std: 1")
    print(f"\n   Thống kê sau Standard Scaling:")
    print(df_standardized[numeric_cols].describe())
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Original
    for col in numeric_cols:
        axes[0].hist(df_clean[col].dropna(), alpha=0.7, label=col, bins=30)
    axes[0].set_title('Original Data')
    axes[0].set_xlabel('Value')
    axes[0].set_ylabel('Frequency')
    axes[0].legend()
    
    # Min-Max
    for col in numeric_cols:
        axes[1].hist(df_minmax[col].dropna(), alpha=0.7, label=col, bins=30)
    axes[1].set_title('Min-Max Scaled [0,1]')
    axes[1].set_xlabel('Value')
    axes[1].set_ylabel('Frequency')
    axes[1].legend()
    
    # Standard
    for col in numeric_cols:
        axes[2].hist(df_standardized[col].dropna(), alpha=0.7, label=col, bins=30)
    axes[2].set_title('Standard Scaled (Mean=0, Std=1)')
    axes[2].set_xlabel('Value')
    axes[2].set_ylabel('Frequency')
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n💡 Chọn chuẩn hóa:")
    print(f"   • Min-Max: Khi cần giữ ngữ cảnh của dữ liệu (VD: Neural Networks)")
    print(f"   • Standard: Khi dữ liệu theo phân phối chuẩn (VD: Linear Regression)")
else:
    print("\n⚠️ Không có cột số để chuẩn hóa")
    df_minmax = df_clean
    df_standardized = df_clean

## BLOCK 14: Chọn phiên bản cuối cùng

In [ ]:
print("="*70)
print("CHỌN PHIÊN BẢN CUỐI CÙNG")
print("="*70)

print(f"\n📌 Có 3 phiên bản dữ liệu sẵn sàng:")
print(f"   1. df_clean - Dữ liệu đã làm sạch (gốc)")
print(f"   2. df_minmax - Dữ liệu đã chuẩn hóa Min-Max [0,1]")
print(f"   3. df_standardized - Dữ liệu đã chuẩn hóa Standard (mean=0, std=1)")

# ========== CẤU HÌNH CHỌN PHIÊN BẢN ==========
VERSION = 'clean'  # Chọn: 'clean', 'minmax', 'standard'

print(f"\n➜ Phiên bản được chọn: {VERSION}")

if VERSION == 'clean':
    df_final = df_clean
elif VERSION == 'minmax':
    df_final = df_minmax
elif VERSION == 'standard':
    df_final = df_standardized
else:
    print("⚠️ Phiên bản không hợp lệ, sử dụng df_clean")
    df_final = df_clean

print(f"\n✓ Dữ liệu cuối cùng:")
print(f"   • Số dòng: {len(df_final):,}")
print(f"   • Số cột: {len(df_final.columns)}")
print(f"\n   Dữ liệu mẫu:")
print(df_final.head())

## BLOCK 15: Lưu dữ liệu đã xử lý

In [ ]:
print("="*70)
print("LƯU DỮ LIỆU ĐÃ XỬ LÝ")
print("="*70)

# ========== LƯU CÁC PHIÊN BẢN ==========

# 1. Lưu dữ liệu đã làm sạch (gốc)
print(f"\n1️⃣ Lưu dữ liệu đã làm sạch...")
try:
    df_clean.to_excel(output_clean, index=False)
    print(f"   ✓ {output_clean}")
    df_clean.to_csv(output_clean_csv, index=False, encoding='utf-8-sig')
    print(f"   ✓ {output_clean_csv}")
except Exception as e:
    print(f"   ❌ Lỗi: {e}")

# 2. Lưu dữ liệu đã chuẩn hóa Min-Max
print(f"\n2️⃣ Lưu dữ liệu đã chuẩn hóa Min-Max...")
try:
    output_minmax = output_normalized.replace('.xlsx', '_minmax.xlsx')
    df_minmax.to_excel(output_minmax, index=False)
    print(f"   ✓ {output_minmax}")
except Exception as e:
    print(f"   ❌ Lỗi: {e}")

# 3. Lưu dữ liệu đã chuẩn hóa Standard
print(f"\n3️⃣ Lưu dữ liệu đã chuẩn hóa Standard...")
try:
    output_standard = output_normalized.replace('.xlsx', '_standard.xlsx')
    df_standardized.to_excel(output_standard, index=False)
    print(f"   ✓ {output_standard}")
except Exception as e:
    print(f"   ❌ Lỗi: {e}")

# 4. Lưu báo cáo metadata
print(f"\n4️⃣ Lưu báo cáo metadata...")
try:
    metadata = pd.DataFrame({
        'Thông Tin': [
            'Dữ liệu gốc (dòng)',
            'Dữ liệu gốc (cột)',
            'Dữ liệu sạch (dòng)',
            'Dữ liệu sạch (cột)',
            'Dòng xóa',
            'Cột xóa',
            'Ngày xử lý'
        ],
        'Giá Trị': [
            len(df_original),
            len(df_original.columns),
            len(df_clean),
            len(df_clean.columns),
            len(df_original) - len(df_clean),
            len(df_original.columns) - len(df_clean.columns),
            datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        ]
    })
    output_metadata = output_clean.replace('.xlsx', '_metadata.xlsx')
    metadata.to_excel(output_metadata, index=False)
    print(f"   ✓ {output_metadata}")
except Exception as e:
    print(f"   ❌ Lỗi: {e}")

print(f"\n" + "="*70)
print(f"✓ Tất cả file đã được lưu thành công!")
print(f"="*70)

## BLOCK 16: Báo cáo tóm tắt cuối cùng

In [ ]:
print("\n" + "="*70)
print("📋 BÁO CÁO TÓM TẮT CUỐI CÙNG")
print("="*70)

print(f"\n1️⃣ THÔNG TIN TỔNG QUÁT:")
print("-" * 70)
print(f"   • Dữ liệu gốc: {len(df_original):,} dòng × {len(df_original.columns)} cột")
print(f"   • Dữ liệu sạch: {len(df_clean):,} dòng × {len(df_clean.columns)} cột")
print(f"   • Dòng bị xóa: {len(df_original) - len(df_clean):,} ({(len(df_original) - len(df_clean))/len(df_original)*100:.2f}%)")
print(f"   • Cột bị xóa: {len(df_original.columns) - len(df_clean.columns)}")

print(f"\n2️⃣ CÁC BƯỚC XỬ LÝ:")
print("-" * 70)
print(f"   ✓ Loại bỏ khoảng trắng dư thừa (text)")
print(f"   ✓ Loại bỏ dòng trùng lặp")
print(f"   ✓ Xử lý missing values")
print(f"   ✓ Phát hiện outliers (IQR method)")
if REMOVE_OUTLIERS:
    print(f"   ✓ Xóa outliers")
else:
    print(f"   • Giữ lại outliers")
print(f"   ✓ Chuẩn hóa dữ liệu")

print(f"\n3️⃣ KIỂU DỮ LIỆU:")
print("-" * 70)
dtype_counts = df_clean.dtypes.value_counts()
for dtype, count in dtype_counts.items():
    print(f"   • {dtype}: {count}")

print(f"\n4️⃣ THỐNG KÊ MISSING VALUES:")
print("-" * 70)
total_missing = df_clean.isnull().sum().sum()
total_cells = len(df_clean) * len(df_clean.columns)
print(f"   • Missing cells: {total_missing:,} / {total_cells:,}")
print(f"   • Tỷ lệ missing: {total_missing/total_cells*100:.4f}%")

print(f"\n5️⃣ FILE ĐẦU RA:")
print("-" * 70)
print(f"   📄 {output_clean}")
print(f"   📄 {output_clean_csv}")
print(f"   📄 {output_minmax}")
print(f"   📄 {output_standard}")
print(f"   📄 {output_metadata}")

print(f"\n6️⃣ ĐỐI TƯỢNG DỮ LIỆU MẪU (5 DÒNG ĐẦU):")
print("-" * 70)
print(df_clean.head())

print(f"\n7️⃣ ĐỐI TƯỢNG DỰA LIỆU MẪU (5 DÒNG CUỐI):")
print("-" * 70)
print(df_clean.tail())

print(f"\n" + "="*70)
print(f"✅ HOÀN THÀNH! Quá trình làm sạch và chuẩn hóa dữ liệu thành công!")
print(f"="*70)
print(f"\n🎉 Dữ liệu đã sẵn sàng cho giai đoạn tiếp theo (EDA, Modeling)")